In [ ]:
import torch
from PIL import Image
import chromadb
import os
import base64
from dotenv import load_dotenv
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer

# 從 .env 載入機密金鑰，避免硬編碼在程式碼中
load_dotenv()

# Gemini 有兩把金鑰以分散用量限制：主要金鑰 / 備用金鑰
for _key in ("GEMINI_API_KEY_PRIMARY", "GEMINI_API_KEY_SECONDARY"):
    if not os.environ.get(_key):
        raise RuntimeError(f"缺少環境變數 {_key}，請在 .env 檔中設定")

gemini_api_key_primary = os.environ["GEMINI_API_KEY_PRIMARY"]
gemini_api_key_secondary = os.environ["GEMINI_API_KEY_SECONDARY"]

clientmain = genai.Client(api_key=gemini_api_key_primary)
client = genai.Client(api_key=gemini_api_key_secondary)
vision_model = "gemini-3.1-flash-lite"

def get_user_collection(user_id: str):
    chroma_client = chromadb.PersistentClient(path=f"./vectorstores/{user_id}")
    collection = chroma_client.get_or_create_collection(user_id)
    return collection

# 有 GPU 就用 GPU，否則退回 CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用裝置: {device}")

# 載入檢索用嵌入模型 intfloat/multilingual-e5-base（繁中+英文皆強，跨語言檢索佳），
# 取代原本對中文較弱且只看前 77 個 token 的 CLIP 文字編碼器。
# E5 需要前綴：文件用 "passage: "，查詢用 "query: "。
# 只載入一次，避免重複執行 cell 時重新建立模型。
EMBED_MODEL = "intfloat/multilingual-e5-base"
if "embedder" not in globals():
    embedder = SentenceTransformer(EMBED_MODEL, device=device)

# Embed 文字（passage 端：文件 / 圖片描述）
def embed_text(texts: list[str]):
    inputs = [f"passage: {t}" for t in texts]
    return embedder.encode(inputs, normalize_embeddings=True).tolist()

# Embed 圖片
def describe_image(image_path: str) -> str:
    """用 Gemini 產生圖片的文字描述"""
    with open(image_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")
    
    ext = image_path.split(".")[-1].lower()
    mime = "image/jpeg" if ext in ["jpg", "jpeg"] else f"image/{ext}"
    
    response = client.models.generate_content(
        model=vision_model,
        contents=[
            types.Part.from_bytes(data=base64.b64decode(image_data), mime_type=mime),
            types.Part.from_text(text="""請詳細描述這張圖片中有什麼內容。注意: 1.不要使用粗體 2.圖片應該以左右反轉的方式描述 3. 在開頭輸出
            你判斷這是什麼科目(國文、英文、數學、物理、化學、地球科學、生物、地理、公民、歷史擇一)的筆記，格式為: 這是一份【科目】科的筆記。
            """
            )
        ],
        config={
            "temperature": 0,
            "top_p": 0.95,
            "top_k": 20,
        }
    )
    return response.text

def add_txt(user_id: str, path: str):
    collection = get_user_collection(user_id)
    file_id = os.path.basename(path)
    
    existing = collection.get(ids=[file_id])
    if existing["ids"]:
        print(f"{file_id} 已存在，跳過")
        return
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    collection.add(
        ids=[file_id],
        embeddings=[embed_text([content])[0]],
        documents=[content],
        metadatas=[{"type": "text", "source": path}]
    )
    print(f"{file_id} 已加入")

def add_image(user_id: str, path: str):
    collection = get_user_collection(user_id)
    file_id = os.path.basename(path)
    existing = collection.get(ids=[file_id])
    if existing["ids"]:
        print(f"{file_id} 已存在，跳過")
        return
    
    description = describe_image(path)
    print(f"圖片描述：{description}")
    
    collection.add(
        ids=[file_id],
        embeddings=[embed_text([description])[0]],  # ← embed 描述文字
        documents=[description],
        metadatas=[{"type": "image", "source": path}]
    )
    print(f"{file_id} 已加入")

def delete_file(user_id: str, filename: str):
    collection = get_user_collection(user_id)
    collection.delete(ids=[filename])

def delete_image(user_id: str, path: str):
    collection = get_user_collection(user_id)
    file_id = os.path.basename(path)
    collection.delete(ids=[file_id])
    os.remove(path)